# Sleep-EDF: серия 058–098

1. Подключите full_psg_v1 и прежние ZIP с 019/041/053 (желательно с predictions). Для D3/D4/E нужен старый полный cache/features на train/val/test.
2. Internet On, GPU T4×2. Код должен быть опубликован по CODE_REF. Выберите RUN_GROUP. Первый запуск: SMOKE_ONLY=True.
3. После успешного smoke поставьте SMOKE_ONLY=False и Run All. Готовые smoke/full переиспользуются. C/D/E/F требуют ALLOW_TRAINING=True.
4. Удобный порядок: frozen → ensembles → tiny_finetune → embeddings → specialists. sleepgpt и adaptive запускайте отдельно: они существенно тяжелее.
5. Для всех 41 конфигураций: RUN_GROUP="all", ALLOW_TRAINING=True. Для отдельных ID: custom + CUSTOM_IDS. Полные модели запускаются только после их успешного smoke.
6. После прерывания выполните экспорт последней ячейкой. В новой сессии распакуйте результаты и веса в отдельные каталоги Input и задайте RESUME_ROOT / RESUME_TRAINED_ROOT. Используйте ту же версию кода.

Одна GPU на процесс. Две T4 не объединяются в 32 GiB. Кэши и официальные веса находятся в /tmp, компактный ZIP их не содержит. Best/last и classifier.joblib экспортируются отдельно. Независимые ошибки не останавливают оставшуюся группу.

SleepGPT использует настоящий pretrained encoder и новую обучаемую голову; это не zero-shot. SleepFM-2 не получает фиктивной конфигурации: официальный публичный репозиторий недоступен при проверке.
Training smoke использует реальные окна около начала сна по train labels, чтобы проверить все пять классов. Полный train не обрезается; smoke не оценивает качество.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/revlze/sleep-edf-classification.git"
CODE_REF = "feat/pretrained-expansion"  # После публикации можно закрепить SHA.
PROJECT_ROOT = Path("/tmp/sleep-edf-expansion-code")
OUTPUT_ROOT = Path("/kaggle/working/artefacts")
CHECKPOINT_CACHE = Path("/tmp/sleep-edf-expansion-checkpoints")
CACHE_ROOT = Path("/tmp/sleep-edf-expansion-cache")
TRAINED_ROOT = Path("/kaggle/working/trained-checkpoints")
CHECKPOINT_ROOT = None  # Необязательный Input с официальными файлами весов.
RESULT_INPUTS = []  # Автопоиск прежних ZIP/каталогов с 019/041/053 и predictions.
FEATURE_CACHE = (
    None  # Для D3/D4/E: каталог старого ПОЛНОГО cache/features с definition.json.
)
RESUME_ROOT = None  # Прошлый распакованный OUTPUT_ROOT, та же версия кода.
RESUME_TRAINED_ROOT = None  # Прошлый распакованный архив best/last + classifier.joblib.

RUN_GROUP = "frozen"  # frozen / ensembles / inference / tiny_finetune / embeddings / specialists / sleepgpt / adaptive / all / custom
CUSTOM_IDS = []  # Например ["082_pretrained", "083_pretrained"].
SMOKE_ONLY = True  # Сначала True; затем False: smoke -> полный прогон успешных smoke.
SMOKE_EPOCHS = 399  # Проверяет и внутренние окна длинного context200.
ALLOW_TRAINING = (
    False  # True для tiny_finetune / embeddings / specialists / sleepgpt / all.
)
EXPORT_TRAINED_CHECKPOINTS = True
DEVICE = (
    "cuda:0"  # Одна T4 16GiB на процесс. Группу можно отдельно запустить на cuda:1.
)
THREADS = max(1, (os.cpu_count() or 2) - 1)

GROUPS = {
    "frozen": list(range(58, 64)),
    "ensembles": list(range(64, 82)),
    "inference": list(range(58, 82)),
    "tiny_finetune": list(range(82, 87)),
    "embeddings": list(range(87, 92)),
    "specialists": list(range(92, 96)),
    "sleepgpt": [96, 97],
    "adaptive": [98],
    "all": list(range(58, 99)),
}
CONFIG_IDS = (
    CUSTOM_IDS
    if RUN_GROUP == "custom"
    else [f"{i:03d}_pretrained" for i in GROUPS[RUN_GROUP]]
)
if not CONFIG_IDS or any(
    not isinstance(name, str) or not name.endswith("_pretrained") for name in CONFIG_IDS
):
    raise ValueError("Нужен список полных ID, например 058_pretrained")
if any(82 <= int(name[:3]) <= 97 for name in CONFIG_IDS) and not ALLOW_TRAINING:
    raise RuntimeError(
        "Эти конфигурации обучаются на train: требуется ALLOW_TRAINING=True"
    )


def checkout_code():
    if PROJECT_ROOT.exists():
        current_ref = subprocess.check_output(
            ["git", "config", "sleepedf.bootstrapRef"], cwd=PROJECT_ROOT, text=True
        ).strip()
        if current_ref != CODE_REF:
            raise RuntimeError(
                "Для другого CODE_REF задайте новый PROJECT_ROOT в /tmp."
            )
        print(
            "Используется существующий checkout; для обновления ветки задайте новый PROJECT_ROOT."
        )
        return
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory(prefix="sleep-git-") as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\nimport os, sys\n"
            'print("x-access-token" if "Username" in sys.argv[1] else os.environ["SLEEP_GITHUB_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        env = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "SLEEP_GITHUB_TOKEN": token,
        }
        subprocess.run(
            ["git", "init", "--quiet", "--initial-branch=bootstrap", str(PROJECT_ROOT)],
            check=True,
        )
        subprocess.run(
            ["git", "remote", "add", "origin", REPO_URL], cwd=PROJECT_ROOT, check=True
        )
        subprocess.run(
            [
                "git",
                "-c",
                "credential.helper=",
                "fetch",
                "--quiet",
                "--depth",
                "1",
                "origin",
                CODE_REF,
            ],
            cwd=PROJECT_ROOT,
            env=env,
            check=True,
        )
        subprocess.run(
            ["git", "checkout", "--quiet", "--detach", "FETCH_HEAD"],
            cwd=PROJECT_ROOT,
            check=True,
        )
        subprocess.run(
            ["git", "config", "sleepedf.bootstrapRef", CODE_REF],
            cwd=PROJECT_ROOT,
            check=True,
        )
    del token


checkout_code()
commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
print("Код:", commit)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook import run_command  # noqa: E402

# Устанавливаем только инструмент uv; зависимости проекта устанавливает uv.lock.

UV_VERSION = "0.12.15"
if shutil.which("uv") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"uv=={UV_VERSION}"], check=True
    )
UV = shutil.which("uv")
ENV = {
    "UV_PROJECT_ENVIRONMENT": "/tmp/sleep-edf-expansion-venv",
    "UV_CACHE_DIR": "/tmp/sleep-pretrained-uv-cache",
    "UV_PYTHON_INSTALL_DIR": "/tmp/sleep-pretrained-python",
    "PYTHONPATH": str(PROJECT_ROOT),
    "MPLBACKEND": "Agg",  # Backend только для дочернего CLI, не Kaggle-ядра.
}
UV_PROJECT = PROJECT_ROOT / "environments/pretrained"
print("Ядро notebook:", sys.version)
subprocess.run(["nvidia-smi"], check=True)
run_command(
    [
        UV,
        "sync",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "--python",
        "3.12.12",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)
BASE = [
    UV,
    "run",
    "--project",
    str(UV_PROJECT),
    "--locked",
    "--no-dev",
    "python",
    "-m",
    "src.expansion_run",
]
run_command(
    [
        UV,
        "run",
        "--project",
        str(UV_PROJECT),
        "--locked",
        "--no-dev",
        "python",
        "-c",
        "from src.pretrained import hardware; import json; print(json.dumps(hardware("
        + repr(DEVICE)
        + "),indent=2))",
    ],
    cwd=PROJECT_ROOT,
    env=ENV,
)

In [ ]:
from zipfile import ZipFile

import pandas as pd
from IPython.display import display

roots = [
    p.parent
    for p in Path("/kaggle/input").rglob("MANIFEST.json")
    if (p.parent / "labels.csv").is_file()
    and (p.parent / "epochs/train").is_dir()
    and (p.parent / "epochs/val").is_dir()
    and (p.parent / "epochs/test").is_dir()
]
if len(roots) != 1:
    raise RuntimeError(f"Нужен один исходный full_psg_v1; найдено {roots}")
DATA_ROOT = roots[0]
if not RESULT_INPUTS:
    candidates = [
        p.parent.parent
        for p in Path("/kaggle/input").rglob("val_predictions.npz")
        if p.parent.name in ("019_pretrained", "041_pretrained", "053_pretrained")
        and "smoke" not in p.parts
    ]
    for candidate in Path("/kaggle/input").rglob("*.zip"):
        try:
            with ZipFile(candidate) as zipped:
                if any(
                    name.endswith(
                        (
                            "019_pretrained/val_predictions.npz",
                            "041_pretrained/val_predictions.npz",
                            "053_pretrained/val_predictions.npz",
                        )
                    )
                    or name.startswith("predictions/")
                    for name in zipped.namelist()
                ):
                    candidates.append(candidate)
        except Exception as error:
            print("Не прочитан Input ZIP:", candidate.name, str(error))
    RESULT_INPUTS = sorted(set(candidates))
if FEATURE_CACHE is None:
    definitions = []
    for p in Path("/kaggle/input").rglob("definition.json"):
        try:
            d = json.loads(p.read_text())
            if (
                d.get("version") == "linear106_v1"
                and d.get("revision") == "full_psg_v1"
                and not d.get("smoke")
            ):
                definitions.append(p.parent)
        except (ValueError, OSError):
            pass
    if len(definitions) == 1:
        FEATURE_CACHE = definitions[0]
# Можно подключить старый LR ZIP целиком: читаем только полный 106-feature cache.
if FEATURE_CACHE is None:
    feature_archives = []
    for candidate in Path("/kaggle/input").rglob("*.zip"):
        try:
            with ZipFile(candidate) as zipped:
                for name in zipped.namelist():
                    if name.endswith("/definition.json"):
                        definition = json.loads(zipped.read(name))
                        if (
                            definition.get("version") == "linear106_v1"
                            and definition.get("revision") == "full_psg_v1"
                            and not definition.get("smoke")
                        ):
                            feature_archives.append((candidate, name))
        except (ValueError, OSError):
            pass
    if len(feature_archives) == 1:
        candidate, definition_name = feature_archives[0]
        import hashlib

        key = hashlib.sha256((str(candidate) + definition_name).encode()).hexdigest()[
            :16
        ]
        target = Path("/tmp/sleep-edf-imported-106") / key
        target.mkdir(parents=True, exist_ok=True)
        prefix = str(Path(definition_name).parent).replace("\\", "/") + "/"
        with ZipFile(candidate) as zipped:
            for name in zipped.namelist():
                if (
                    name.startswith(prefix)
                    and "/" not in name[len(prefix) :]
                    and (name == definition_name or name.endswith("_epoch.npz"))
                ):
                    destination = target / Path(name).name
                    if not destination.exists():
                        temporary = destination.with_suffix(destination.suffix + ".tmp")
                        with zipped.open(name) as source, temporary.open("wb") as out:
                            shutil.copyfileobj(source, out)
                        temporary.replace(destination)
        FEATURE_CACHE = target
        print("106-feature cache извлечён из Input ZIP:", candidate.name)
    elif len(feature_archives) > 1:
        print(
            "Найдено несколько полных feature cache в ZIP; укажите FEATURE_CACHE после распаковки нужного."
        )

for previous, target in [
    (RESUME_ROOT, OUTPUT_ROOT),
    (RESUME_TRAINED_ROOT, TRAINED_ROOT),
]:
    if previous is not None:
        if not target.exists():
            shutil.copytree(Path(previous), target)
        else:
            print("Продолжаем существующий каталог:", target)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not RESULT_INPUTS:
    print(
        "Input с прежними вероятностями не найден: baseline будет рассчитан и закэширован. Можно остановить и подключить старый ZIP, чтобы сэкономить время."
    )
if FEATURE_CACHE is None and any(
    i in CONFIG_IDS
    for i in [
        "089_pretrained",
        "090_pretrained",
        *[f"{j:03d}_pretrained" for j in range(92, 96)],
    ]
):
    print(
        "D3/D4/E требуют старый полный 106-feature cache, включая test; без него только эти конфигурации завершатся явной ошибкой."
    )
COMMON = [
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--cache-root",
    str(CACHE_ROOT),
    "--checkpoint-cache",
    str(CHECKPOINT_CACHE),
    "--trained-root",
    str(TRAINED_ROOT),
    "--device",
    DEVICE,
    "--threads",
    str(THREADS),
]
for previous in RESULT_INPUTS:
    COMMON += ["--result-input", str(previous)]
if FEATURE_CACHE is not None:
    COMMON += ["--feature-cache", str(FEATURE_CACHE)]
if CHECKPOINT_ROOT is not None:
    COMMON += ["--checkpoint-root", str(CHECKPOINT_ROOT)]
if ALLOW_TRAINING:
    COMMON += ["--allow-training"]

rows = []
for name in CONFIG_IDS:
    config_path = PROJECT_ROOT / "configs" / f"{name}.yaml"
    if not config_path.is_file():
        raise FileNotFoundError(
            f"{config_path}: сначала опубликуйте актуальный код по CODE_REF"
        )
    state = OUTPUT_ROOT / ("smoke" if SMOKE_ONLY else "") / name
    rows.append(
        {
            "ID": name,
            "status": "complete"
            if (state / "metrics.json").exists()
            else "partial"
            if state.exists()
            else "pending",
        }
    )
display(pd.DataFrame(rows))
print(
    "DATA_ROOT:",
    DATA_ROOT,
    "\nRESULT_INPUTS:",
    [str(p) for p in RESULT_INPUTS],
    "\nFEATURE_CACHE:",
    FEATURE_CACHE,
)

In [ ]:
def run_batch(names, smoke):
    if not names:
        return []
    command = [
        *BASE,
        *[str(PROJECT_ROOT / "configs" / f"{name}.yaml") for name in names],
        *COMMON,
    ]
    if smoke:
        command += ["--smoke", "--smoke-epochs", str(SMOKE_EPOCHS)]
    import time

    started_ns = time.time_ns()
    process_failed = False
    try:
        run_command(command, cwd=PROJECT_ROOT, env=ENV)
    except subprocess.CalledProcessError:
        process_failed = True
        print(
            "Есть ошибки или прерывание. Завершённые результаты сохранены, ошибки находятся в <ID>/last_error.json."
        )
    namespace = OUTPUT_ROOT / "smoke" if smoke else OUTPUT_ROOT
    failure_file = namespace / "expansion_failures.json"
    fresh = failure_file.exists() and failure_file.stat().st_mtime_ns >= started_ns
    if process_failed and not fresh:
        print(
            "Процесс завершился до итогового отчёта; полный запуск этой группы пропущен."
        )
        return []
    failures = json.loads(failure_file.read_text()) if fresh else []
    failed_ids = {row["experiment_id"] for row in failures}
    passed = [
        name
        for name in names
        if name not in failed_ids and (namespace / name / "metrics.json").is_file()
    ]
    if process_failed and not passed:
        print("Успешных результатов этого режима пока нет.")
    return passed


passed = run_batch(CONFIG_IDS, smoke=True)
if not SMOKE_ONLY:
    run_batch(passed, smoke=False)
namespace = OUTPUT_ROOT / "smoke" if SMOKE_ONLY else OUTPUT_ROOT
rows = []
for name in CONFIG_IDS:
    path = namespace / name / "metrics.json"
    if path.is_file():
        m = json.loads(path.read_text())
        rows.append(
            {
                "ID": name,
                "mode": "SMOKE" if m["smoke"] else "VAL+TEST",
                "val_macro_f1": m["validation"]["macro_f1"],
                "training_s": m["training_seconds"],
                "inference_s": round(m["inference_seconds"], 1),
                "peak_GPU_GiB": round(m["peak_gpu_bytes"] / 2**30, 2)
                if m["peak_gpu_bytes"] is not None
                else None,
            }
        )
    else:
        error_path = namespace / name / "last_error.json"
        rows.append(
            {
                "ID": name,
                "mode": "FAILED/PARTIAL",
                "error": json.loads(error_path.read_text())["error"]
                if error_path.exists()
                else "not completed",
            }
        )
display(pd.DataFrame(rows))

Экспорт можно выполнить и после прерывания ячейки обучения. Скачайте **оба ZIP**, если было обучение: результаты и отдельно веса для resume. В архив результатов входят журнал, конфигурации, снимок кода, вероятности и submissions. Smoke не записывается в experiments.csv.

In [ ]:
from datetime import UTC, datetime
from zipfile import ZIP_DEFLATED, ZipFile

from IPython.display import FileLink, display

stamp = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
archive = Path("/kaggle/working") / f"sleep-edf-expansion-{stamp}.zip"
run_command(
    [*BASE, "--output-root", str(OUTPUT_ROOT), "--export", str(archive)],
    cwd=PROJECT_ROOT,
    env=ENV,
)
display(
    FileLink(
        str(archive.relative_to(Path.cwd())),
        result_html_prefix="Результаты, вероятности и submissions: ",
    )
)
if ALLOW_TRAINING and EXPORT_TRAINED_CHECKPOINTS and TRAINED_ROOT.exists():
    weights_zip = Path("/kaggle/working") / f"sleep-edf-trained-{stamp}.zip"
    with ZipFile(weights_zip, "x", compression=ZIP_DEFLATED) as output:
        for path in sorted(TRAINED_ROOT.rglob("*")):
            if (
                path.is_file()
                and path.suffix in (".pt", ".json", ".joblib")
                and ".tmp." not in path.name
            ):
                output.write(path, path.relative_to(TRAINED_ROOT).as_posix())
    display(
        FileLink(
            str(weights_zip.relative_to(Path.cwd())),
            result_html_prefix="Отдельно best/last checkpoints для продолжения обучения: ",
        )
    )